In [800]:
import sys
sys.path.append("../") # go to parent dir

from custom_helpers_py.utilities import camel_to_snake_case
import pandas as pd
from os import listdir
from os.path import join
import json

In [801]:
COMPRESSED_FOLDER_PATH = join("../.outFiles/analysis/compressed")
POLITICIANS_FP = join(COMPRESSED_FOLDER_PATH, "politician.csv")

politicians_df = pd.read_csv(POLITICIANS_FP, encoding="utf-8")
politicians_df

,p_full_name,p_chamber,p_state,p_district,p_party
0,Ralph Lee Abraham,H,Louisiana,5,R
1,Alma S Adams,H,North Carolina,12,D
2,Robert B Aderholt,H,Alabama,4,R
3,Pete Aguilar,H,California,33,D
4,Lamar Alexander,S,Tennessee,NaN,R
...,...,...,...,...,...
958,David Young,H,Iowa,3,R
959,Don Young,H,Alaska,At Large,R
960,Todd Young,S,Indiana,NaN,R
961,Lee M Zeldin,H,New York,1,R


In [802]:
from difflib import SequenceMatcher

def is_str(to_test):
    return isinstance(to_test, str)

def get_similarity_ratio(a: str, b:str):
    return SequenceMatcher(None, a, b).ratio()

KNOWN_NAME_FIX_DICT = {
    "WILLIAM M CASSIDY": "Bill Cassidy",
    "PAUL RYAN": "Paul D Ryan",
    "NICK J II RAHALL": "Nick J Rahall",
    "EARL LEROY CARTER": "Earl L \"Buddy\" Carter",
    "NICHOLAS V TAYLOR": "Van Taylor",
    "FACS DUNN": "Neal P Dunn",
    "RODNEY LELAND BLUM": "Rod Blum",
    "TJ JOHN (TJ) COX": "TJ Cox",
    "ROBERT P CORKER JR": "Bob Corker",
    "RAFAEL E CRUZ": "Ted Cruz",
    "JOHN F REED": "Jack Reed",
    "ROBERT P CORKER JR": "Bob Corker",
    "RODNEY LELAND BLUM": "Rod Blum",
    "JOHN F REED": "Jack Reed",
    "JAMES E BANKS": "Jim Banks",
    "BOB LATTA": "Robert E Latta",
    "JAMES VANCE": "J D Vance",
    "CHRISTOPHER L JACOBS": "Chris Jacobs",
    "MICHAEL D CRAPO": "Mike Crapo",
    "MIKE SIMPSON": "Michael K Simpson"
}

def fix_full_name_in_df(in_df: pd.DataFrame) -> pd.DataFrame:
    correct_list_df = politicians_df[["p_full_name"]].drop_duplicates(subset="p_full_name")
    correct_list_df["p_full_name_upper"] =  correct_list_df["p_full_name"].str.upper()

    correct_list = correct_list_df[["p_full_name", "p_full_name_upper"]].to_dict(orient="records")

    correct_dict = {}
    for obj in correct_list:
        lower = obj["p_full_name"]
        upper = obj["p_full_name_upper"]
        correct_dict[upper] = lower
    
    def fix_in_name(in_row: dict):
        in_str:str = in_row["p_full_name"]
        original_name = in_str

        if not is_str(in_str):
            return pd.NA

        in_name = in_str.upper()

        TITLE_LIST = [
            "MR",
            "MS",
            "MRS",
            "DR",
            "MD",
            "HONORABLE",
            "HON"
        ]
        for title in TITLE_LIST:
            to_test = title + "-"
            in_name = in_name.replace(to_test, "")

            to_test = "-" + title
            in_name = in_name.replace(to_test, "")

            to_test = " " +  title + " "
            in_name = in_name.replace(to_test, " ")

            to_test = " " + title
            in_name = in_name.replace(to_test, "")
        
        in_name = in_name.replace(".", "")
        in_name = in_name.replace("-", " ")

        # Start mapping
        known = KNOWN_NAME_FIX_DICT.get(in_name, None)
        if known:
            return pd.Series({"p_full_name": original_name, "fix_name": known, "fix_confidence": 1})

        correction = correct_dict.get(in_name, None)
        if correction:
            return pd.Series({"p_full_name": original_name,"fix_name": correction, "fix_confidence": 1})
        
        # Approximate
        best_ratio = 0
        best_match = None
        for c_obj in correct_list:
            fn, fn_upper = c_obj["p_full_name"], c_obj["p_full_name_upper"]

            sim_ratio = get_similarity_ratio(in_name, fn_upper)
            if sim_ratio > best_ratio:
                best_ratio = sim_ratio
                best_match = fn
        
        if best_ratio > 0.5:
            return pd.Series({"p_full_name": original_name,"fix_name": best_match, "fix_confidence": best_ratio})
        return pd.Series({"p_full_name": original_name, "fix_name": pd.NA, "fix_confidence": 0})
    
    fix_df = in_df[["p_full_name"]].drop_duplicates(subset="p_full_name")
    fix_df: pd.DataFrame = fix_df.apply(fix_in_name, axis=1)

    in_df = in_df.merge(fix_df, on="p_full_name", how="left")
    in_df = in_df.rename(columns={
        "p_full_name": "old_p_full_name",
        "fix_name": "p_full_name",
        "fix_confidence": "fix_name_confidence"
    })
    return in_df

def merge_with_politicians_df(in_df: pd.DataFrame):
    in_df = in_df.merge(politicians_df, on="p_full_name", how="left", suffixes=("_old", ""))
    in_df = in_df.drop(columns=[col for col in in_df.columns if col.endswith("_old")])
    return in_df

def read_compressed_csv(in_name: str):
   return pd.read_csv(join(COMPRESSED_FOLDER_PATH, in_name + ".csv"), encoding="utf-8")  

In [803]:
# Fix self house
self_house_df = read_compressed_csv("self_house") 

def fix_asset_desc(in_desc: str):
   if not is_str(in_desc):
      return pd.NA

   in_desc = in_desc.replace("FILING STATUS NEW", "")
   return in_desc.strip()

self_house_df["asset_desc"] = self_house_df["asset_desc"].apply(fix_asset_desc)


self_house_df  = fix_full_name_in_df(self_house_df)
self_house_df = merge_with_politicians_df(self_house_df)
self_house_df = self_house_df.fillna(pd.NA)
   
self_house_df


,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,old_p_full_name,tx_year,doc_id,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,DECATUR ALA CITY BRD ED SPL TAX SCH WTS,<NA>,<NA>,P,JT,07/3/2014,07/3/2014,$15001 - $50000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
1,ETOWAH CNTY ALA BRD ED CAP OUTLAY WTS,<NA>,<NA>,P,JT,04/11/2014,04/11/2014,$1001 - $15000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
2,MOBILE COUNTY ALA BRD SCH COMMRSCAP OUTLAY WTS,<NA>,<NA>,P,JT,04/16/2014,04/16/2014,$1001 - $15000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
3,MORGAN STANLEY CAP TR V GTD CAP SECS (MWO),MWO,<NA>,S,JT,05/5/2014,05/5/2014,$1001 - $15000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
4,PHENIX CITY AL SCH WTS GENL OBLIG- AT,<NA>,<NA>,P,JT,03/28/2014,03/28/2014,$15001 - $50000,,MR-MO BROOKS,2014,20000606,SELF_HOUSE,Mo Brooks,1.0,H,Alabama,5,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
40359,PRINCE GEORGES CNTY MD GO CONSOLIDATED 5.00 DU...,<NA>,GS,S,JT,02/01/2024,02/01/2024,$500001 - $1000000,,SUZAN-K DELBENE,2024,20024495,SELF_HOUSE,Suzan K DelBene,1.0,H,Washington,1,D
40360,BRISTOL-MYERS SQUIBB COMPANY (BMY) | ST | (BMY),BMY,ST,S,<NA>,01/09/2024,01/12/2024,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSEN...,RICK LARSEN,2024,20024339,SELF_HOUSE,Rick Larsen,1.0,H,Washington,2,D
40361,COLGATE-PALMOLIVE COMPANY (CL) | ST |,CL,ST,P,<NA>,01/09/2024,01/12/2024,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSEN...,RICK LARSEN,2024,20024339,SELF_HOUSE,Rick Larsen,1.0,H,Washington,2,D
40362,THE HERSHEY COMPANY (HSY) | ST |,HSY,ST,S,<NA>,01/09/2024,01/12/2024,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSEN...,RICK LARSEN,2024,20024339,SELF_HOUSE,Rick Larsen,1.0,H,Washington,2,D


In [804]:
low_fix_confidence_df = self_house_df[self_house_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence
6352,KENNETH-R BUCK,Ken Buck,0.727273
6657,TOM-THOMAS-JR GRAVES,Tom Graves,0.666667
10262,CHARLIE-JOSEPH CRIST,Charlie Crist,0.787879
18710,NEAL-PATRICK-MD DUNN,Neal P Dunn,0.785714
20280,GREG-FRANCIS MURPHY,Gregory F Murphy,0.742857
20372,DAVID-CHESTON ROUZER,David Rouzer,0.750000
21812,NICHOLAS-VAN TAYLOR,Van Taylor,0.689655
22217,ELIZABETH FLETCHER,Lizzie Fletcher,0.787879
22604,DONALD-STERNOFF-JR BEYER,Donald S Beyer,0.736842
22815,BRYAN-GEORGE STEIL,Bryan Steil,0.758621


In [805]:
# Fix self senate
self_senate_df = read_compressed_csv("self_senate") 

self_senate_df  = fix_full_name_in_df(self_senate_df)
self_senate_df = merge_with_politicians_df(self_senate_df)
self_senate_df = self_senate_df.fillna(pd.NA)
   
self_senate_df


,tx_date,owner,ticker,asset_name,asset_type,tx_type,amount,comments,old_p_full_name,tx_year,doc_id,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,11/11/2014,Spouse,MDLZ,"Mondelez International, Inc. (NASDAQ)",<NA>,Sale (Full),"$50,001 - $100,000",--,ROY BLUNT,2014,9ddbcc76-dc18-4775-a7d5-a1a7063c0ebd,SELF_SENATE,Roy Blunt,1.0,S,Missouri,<NA>,R
1,04/08/2014,Self,AMT,American Tower Corporation (NYSE),<NA>,Sale (Full),"$15,001 - $50,000",--,CORY-A BOOKER,2014,29d797a6-e3ff-4d76-9ee9-2e3840adb15b,SELF_SENATE,Cory A Booker,1.0,S,New Jersey,<NA>,D
2,04/08/2014,Self,NFLX,"Netflix, Inc. (NASDAQ)",<NA>,Sale (Full),"$15,001 - $50,000",--,CORY-A BOOKER,2014,29d797a6-e3ff-4d76-9ee9-2e3840adb15b,SELF_SENATE,Cory A Booker,1.0,S,New Jersey,<NA>,D
3,08/08/2014,Self,NKE,"Nike, Inc. (NYSE)",<NA>,Sale (Full),"$1,001 - $15,000",--,CORY-A BOOKER,2014,7abb2400-6528-4f7f-9739-248dfedc3ca2,SELF_SENATE,Cory A Booker,1.0,S,New Jersey,<NA>,D
4,08/08/2014,Self,IRM,Iron Mountain Inc. (NYSE),<NA>,Sale (Full),"$1,001 - $15,000",--,CORY-A BOOKER,2014,7abb2400-6528-4f7f-9739-248dfedc3ca2,SELF_SENATE,Cory A Booker,1.0,S,New Jersey,<NA>,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18276,12/14/2023,Spouse,SNOW,Snowflake Inc Cl A,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
18277,12/14/2023,Spouse,LLY,Eli Lilly and Company,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
18278,12/07/2023,Self,TGT,Target Corp,Stock,Sale (Full),"$15,001 - $50,000",--,SHELDON WHITEHOUSE,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
18279,12/07/2023,Self,KO,Coca-Cola Company,Stock,Purchase,"$1,001 - $15,000",--,SHELDON WHITEHOUSE,2024,642f38a1-c21c-429e-94c1-86e57ee39de6,SELF_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D


In [806]:
low_fix_confidence_df = self_senate_df[self_senate_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence
1697,JEFFERSON-B SESSIONS-III,Jeff Sessions,0.702703
5305,JOSEPH MANCHIN-III,Joe Manchin,0.758621
6877,TIMOTHY-M KAINE,Tim Kaine,0.750000
8877,A-MITCHELL MCCONNELL-JR,Mitch McConnell,0.789474
15262,WILLIAM-F HAGERTY-IV,Bill Hagerty,0.687500
17318,JOHN-P RICKETTS,Pete Ricketts,0.714286


In [807]:
# Fix watcher house
watcher_house_df = read_compressed_csv("watcher_house") 

# Remove her because she's from puerto rico, not officially a member
watcher_house_df = watcher_house_df[watcher_house_df["p_full_name"] != "Ada Norah Henriquez"]
 
watcher_house_df  = fix_full_name_in_df(watcher_house_df)
watcher_house_df = merge_with_politicians_df(watcher_house_df)
watcher_house_df = watcher_house_df.fillna(pd.NA)
   
watcher_house_df

,notif_date,tx_date,owner,ticker,asset_name,tx_type,amount,old_p_full_name,cap_gains_over_200_usd,asset_industry,asset_sector,doc_id,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,10/04/2021,2021-09-27,joint,BP,BP plc,purchase,"$1,001 - $15,000",Virginia Foxx,False,Integrated oil Companies,Energy,20019557,WATCHER_HOUSE,Virginia Foxx,1.0,H,North Carolina,5,R
1,10/04/2021,2021-09-13,joint,XOM,Exxon Mobil Corporation,purchase,"$1,001 - $15,000",Virginia Foxx,False,Integrated oil Companies,Energy,20019557,WATCHER_HOUSE,Virginia Foxx,1.0,H,North Carolina,5,R
2,10/04/2021,2021-09-10,joint,ILPT,Industrial Logistics Properties Trust - Common...,purchase,"$15,001 - $50,000",Virginia Foxx,False,Real Estate Investment Trusts,Real Estate,20019557,WATCHER_HOUSE,Virginia Foxx,1.0,H,North Carolina,5,R
3,10/04/2021,2021-09-28,joint,PM,Phillip Morris International Inc,purchase,"$15,001 - $50,000",Virginia Foxx,False,Farming/Seeds/Milling,Consumer Non-Durables,20019557,WATCHER_HOUSE,Virginia Foxx,1.0,H,North Carolina,5,R
4,10/04/2021,2021-09-17,self,BLK,BlackRock Inc,sale_partial,"$1,001 - $15,000",Alan S. Lowenthal,False,Investment Bankers/Brokers/Service,Finance,20019570,WATCHER_HOUSE,Alan S Lowenthal,1.0,H,California,47,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
17161,06/10/2020,2020-04-09,--,SWK,"Stanley Black & Decker, Inc.",sale_partial,"$1,001 - $15,000",Ed Perlmutter,False,Diversified Manufacture,Consumer Discretionary,20016738,WATCHER_HOUSE,Ed Perlmutter,1.0,H,Colorado,7,D
17162,06/10/2020,2020-04-09,--,USB,U.S. Bancorp,sale_partial,"$1,001 - $15,000",Ed Perlmutter,False,Major Banks,Finance,20016738,WATCHER_HOUSE,Ed Perlmutter,1.0,H,Colorado,7,D
17163,06/10/2020,2020-03-13,<NA>,BMY,Bristol-Myers Squibb Company,sale_full,"$100,001 - $250,000",Van Taylor,False,Major Pharmaceuticals,Health Care,20016703,WATCHER_HOUSE,Van Taylor,1.0,H,Texas,3,R
17164,06/10/2020,2020-03-13,<NA>,LLY,Eli Lilly and Company,sale_full,"$500,001 - $1,000,000",Van Taylor,False,Major Pharmaceuticals,Health Care,20016703,WATCHER_HOUSE,Van Taylor,1.0,H,Texas,3,R


In [808]:
low_fix_confidence_df = watcher_house_df[watcher_house_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence
895,Michael Patrick Guest,Michael Guest,0.764706
1404,Harold Dallas Rogers,Harold Rogers,0.787879
2987,David Cheston Rouzer,David Rouzer,0.750000
6617,Greg Francis Murphy,Gregory F Murphy,0.742857
6753,Ashley Hinson Arenholz,Ashley Hinson,0.742857
10724,James M. Costa,Jim Costa,0.727273
12800,Felix Barry Moore,Barry Moore,0.785714
13126,Dan Daniel Bishop,Dan Bishop,0.740741


In [809]:
# Fix watcher senate
watcher_senate_df = read_compressed_csv("watcher_senate") 

watcher_senate_df  = fix_full_name_in_df(watcher_senate_df)
watcher_senate_df = merge_with_politicians_df(watcher_senate_df)
watcher_senate_df = watcher_senate_df.fillna(pd.NA)
   
watcher_senate_df

,tx_date,owner,ticker,asset_name,asset_type,tx_type,amount,comments,asset_industry,asset_sector,old_p_full_name,notif_date,doc_id,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,04/18/2023,Spouse,ESS,"Essex Property Trust, Inc. Common Stock",Stock,Sale (Full),"$1,001 - $15,000",--,Real Estate Investment Trusts,Consumer Services,Sheldon Whitehouse,05/17/2023,9fc025a0-f893-47b2-9252-a2820737a409,WATCHER_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
1,04/18/2023,Self,ESS,"Essex Property Trust, Inc. Common Stock",Stock,Sale (Full),"$1,001 - $15,000",--,Real Estate Investment Trusts,Consumer Services,Sheldon Whitehouse,05/17/2023,9fc025a0-f893-47b2-9252-a2820737a409,WATCHER_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
2,05/16/2023,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Michael F. Bennet,05/16/2023,f590a331-1f74-4d08-b79f-88930593f314,WATCHER_SENATE,Michael F Bennet,1.0,S,Colorado,<NA>,D
3,04/04/2023,Spouse,UPS,"United Parcel Service, Inc. Common Stock",Stock,Sale (Full),"$1,001 - $15,000",--,Trucking Freight/Courier Services,Transportation,Shelley Moore Capito,05/15/2023,60987939-a116-41ee-86c2-8ba920461691,WATCHER_SENATE,Shelley Moore Capito,1.0,S,West Virginia,<NA>,R
4,04/04/2023,Spouse,MCD,McDonald's Corporation Common Stock,Stock,Sale (Partial),"$1,001 - $15,000",--,Restaurants,Consumer Services,Shelley Moore Capito,05/15/2023,60987939-a116-41ee-86c2-8ba920461691,WATCHER_SENATE,Shelley Moore Capito,1.0,S,West Virginia,<NA>,R
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8441,08/17/2012,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Sheldon Whitehouse,08/17/2012,5221D7D8-15D0-40FE-A64C-DD242311F0AE,WATCHER_SENATE,Sheldon Whitehouse,1.0,S,Rhode Island,<NA>,D
8442,08/16/2012,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Pat Roberts,08/16/2012,5C666F29-7055-461D-B4F7-5EA73AFCD860,WATCHER_SENATE,Pat Roberts,1.0,S,Kansas,<NA>,R
8443,08/15/2012,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Rob Portman,08/15/2012,0D78FC31-D28A-440A-8E2D-65C4D9861EAB,WATCHER_SENATE,Rob Portman,1.0,S,Ohio,<NA>,R
8444,08/02/2012,<NA>,<NA>,This filing was disclosed via scanned PDF. Use...,PDF Disclosed Filing,<NA>,Unknown,<NA>,<NA>,<NA>,Thomas R. Carper,08/02/2012,CFDE3B80-E8BD-4F2D-9D64-E892C5EFB32A,WATCHER_SENATE,Thomas R Carper,1.0,S,Delaware,<NA>,D


In [810]:
low_fix_confidence_df = watcher_senate_df[watcher_senate_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence


In [811]:
# Fix trendspider
trendspider_df = read_compressed_csv("trendspider") 

def fix_name(in_name: str):
    if "," in in_name:
        tmp = in_name.split(",")
        in_name = tmp[1] + " " + tmp[0]
    in_name = in_name.strip()
    return in_name

trendspider_df["p_full_name"] = trendspider_df["p_full_name"].apply(fix_name)
trendspider_df  = fix_full_name_in_df(trendspider_df)
trendspider_df = merge_with_politicians_df(trendspider_df)
trendspider_df = trendspider_df[trendspider_df["old_p_full_name"] != "James Calhoun"]
trendspider_df = trendspider_df.fillna(pd.NA)
   
trendspider_df

,ticker,asset_name,old_p_full_name,tx_type,amount,tx_date,notif_date,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,NGL,Ngl Energy Partners Lp Common Units Representi...,Mark Dr Green,Sale,"$100,001 - $250,000","Apr 1, 2024","Apr 7, 2024",TREND_SPIDER,Mark E Green,0.909091,H,Tennessee,7,R
1,XNGSY,Een Energy Hldgs Unsp/Adr,Josh Gottheimer,Purchase,"$1,001 - $15,000","Mar 26, 2024","Apr 7, 2024",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
2,AAPL,Apple Inc. - Common Stock,Josh Gottheimer,Sale,"$1,001 - $15,000","Mar 22, 2024","Apr 7, 2024",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
3,SQ,"Block, Inc. Class A Common Stock,",Josh Gottheimer,Purchase,"$1,001 - $15,000","Mar 21, 2024","Apr 7, 2024",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
4,AMD,"Advanced Micro Devices, Inc.",Josh Gottheimer,Purchase,"$1,001 - $15,000","Mar 18, 2024","Apr 7, 2024",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52178,HCA,"Hca Healthcare, Inc.",Josh Gottheimer,Sale,"$1,001 - $15,000","Sep 28, 2022","Oct 16, 2022",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
52179,DXCM,"Dexcom, Inc.",Josh Gottheimer,Sale,"$1,001 - $15,000","Sep 28, 2022","Oct 16, 2022",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
52180,ORCL,Oracle Corporation,Josh Gottheimer,Sale,"$1,001 - $15,000","Sep 28, 2022","Oct 16, 2022",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D
52181,ALNY,"Alnylam Pharmaceuticals, Inc.",Josh Gottheimer,Purchase,"$1,001 - $15,000","Sep 28, 2022","Oct 16, 2022",TREND_SPIDER,Josh Gottheimer,1.000000,H,New Jersey,5,D


In [812]:
low_fix_confidence_df = trendspider_df[trendspider_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df


,old_p_full_name,p_full_name,fix_name_confidence
65,A. Mitchell Jr. McConnell,Mitch McConnell,0.789474
85,Michael Patrick Guest,Michael Guest,0.764706
356,Cindy Axne,Cynthia Axne,0.636364
2385,Donald Sternoff Beyer Jr.,Donald S Beyer,0.736842
3336,David Cheston Rouzer,David Rouzer,0.750000
4209,Felix Barry Moore,Barry Moore,0.785714
4243,Kenneth R. Buck,Ken Buck,0.727273
8409,Michael John Gallagher,Mike Gallagher,0.722222
8418,Ashley Hinson Arenholz,Ashley Hinson,0.742857
13925,Nicholas Van Taylor,Van Taylor,0.689655


In [813]:
# Fix ct
ct_df = read_compressed_csv("ct") 

def fix_name(in_name: str):
    if "," in in_name:
        tmp = in_name.split(",")
        in_name = tmp[1] + " " + tmp[0]
    in_name = in_name.strip()
    return in_name

ct_df["p_full_name"] = ct_df["p_full_name"].apply(fix_name)
ct_df  = fix_full_name_in_df(ct_df)
ct_df = merge_with_politicians_df(ct_df)
ct_df = ct_df[ct_df["old_p_full_name"] != "James Calhoun"]
ct_df = ct_df.fillna(pd.NA)
   
ct_df

,old_p_full_name,asset_name,ticker,notif_date,tx_date,owner,tx_type,amount,avg_ticker_price,data_source,p_full_name,fix_name_confidence,p_chamber,p_state,p_district,p_party
0,Dan Meuser,NVIDIA Corporation,NVDA:US,06:05 Today,20 Feb 2024,Spouse,SELL,250K-500K,694.52,CAPITOL_TRADES,Daniel Meuser,0.869565,H,Pennsylvania,9,R
1,Dan Meuser,NVIDIA Corporation,NVDA:US,06:05 Today,20 Feb 2024,Child,SELL,15K-50K,694.52,CAPITOL_TRADES,Daniel Meuser,0.869565,H,Pennsylvania,9,R
2,Dan Meuser,NVIDIA Corporation,NVDA:US,06:05 Today,20 Feb 2024,Child,SELL,15K-50K,694.52,CAPITOL_TRADES,Daniel Meuser,0.869565,H,Pennsylvania,9,R
3,Dan Meuser,US TREASURY BILLS,<NA>,06:05 Today,23 Feb 2024,Spouse,BUY,250K-500K,<NA>,CAPITOL_TRADES,Daniel Meuser,0.869565,H,Pennsylvania,9,R
4,Nancy Pelosi,FORGE INVESTMENTS LLC,<NA>,06:05 Today,4 Mar 2024,Spouse,BUY,1M-5M,<NA>,CAPITOL_TRADES,Nancy Pelosi,1.000000,H,California,11,D
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
37909,Hal Rogers,BRIDGE BUILDER LARGE CAP VALUE FUND,BBVLX:US,2 Apr 2021,25 Mar 2021,Undisclosed,SELL,1K-15K,15.99,CAPITOL_TRADES,Harold Rogers,0.869565,H,Kentucky,5,R
37910,Hal Rogers,BRIDGE BUILDER SMALL MID CAP VALUE FUND,BBVSX:US,2 Apr 2021,25 Mar 2021,Undisclosed,SELL,1K-15K,14.34,CAPITOL_TRADES,Harold Rogers,0.869565,H,Kentucky,5,R
37911,Mark Green,American Airlines Group Inc,AAL:US,29 Mar 2021,24 Mar 2021,Joint,BUY,1K-15K,21.81,CAPITOL_TRADES,Mark E Green,0.909091,H,Tennessee,7,R
37912,Mark Green,United Airlines Holdings Inc,UAL:US,29 Mar 2021,24 Mar 2021,Joint,BUY,1K-15K,53.83,CAPITOL_TRADES,Mark E Green,0.909091,H,Tennessee,7,R


In [814]:
low_fix_confidence_df = ct_df[ct_df["fix_name_confidence"] < 0.8]
low_fix_confidence_df = low_fix_confidence_df.drop_duplicates("p_full_name")[["old_p_full_name", "p_full_name", "fix_name_confidence"]]

low_fix_confidence_df

,old_p_full_name,p_full_name,fix_name_confidence
44,Bill Keating,William R Keating,0.758621
257,Chuck Fleischmann,"Charles J ""Chuck"" Fleischmann",0.739130
771,Buddy Carter,"Earl L ""Buddy"" Carter",0.727273
794,Brad Schneider,Bradley Scott Schneider,0.756757
837,Don Beyer,Donald S Beyer,0.782609
1947,Gerry Connolly,Gerald E Connolly,0.774194
2845,Jim Himes,James A Himes,0.727273
4678,Jim Langevin,James R Langevin,0.785714
4768,Cindy Axne,Cynthia Axne,0.636364
20893,John Raymond Garamendi,John Garamendi,0.777778


In [815]:
# Create master df
master_df = pd.concat([
    self_house_df,
    watcher_house_df,
    watcher_senate_df,
    ct_df
]).reset_index(drop=True)

master_df

,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,old_p_full_name,...,fix_name_confidence,p_chamber,p_state,p_district,p_party,cap_gains_over_200_usd,asset_industry,asset_sector,comments,avg_ticker_price
0,DECATUR ALA CITY BRD ED SPL TAX SCH WTS,<NA>,<NA>,P,JT,07/3/2014,07/3/2014,$15001 - $50000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
1,ETOWAH CNTY ALA BRD ED CAP OUTLAY WTS,<NA>,<NA>,P,JT,04/11/2014,04/11/2014,$1001 - $15000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
2,MOBILE COUNTY ALA BRD SCH COMMRSCAP OUTLAY WTS,<NA>,<NA>,P,JT,04/16/2014,04/16/2014,$1001 - $15000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
3,MORGAN STANLEY CAP TR V GTD CAP SECS (MWO),MWO,<NA>,S,JT,05/5/2014,05/5/2014,$1001 - $15000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
4,PHENIX CITY AL SCH WTS GENL OBLIG- AT,<NA>,<NA>,P,JT,03/28/2014,03/28/2014,$15001 - $50000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103885,BRIDGE BUILDER LARGE CAP VALUE FUND,BBVLX:US,NaN,SELL,Undisclosed,25 Mar 2021,2 Apr 2021,1K-15K,NaN,Hal Rogers,...,0.869565,H,Kentucky,5,R,NaN,NaN,NaN,NaN,15.99
103886,BRIDGE BUILDER SMALL MID CAP VALUE FUND,BBVSX:US,NaN,SELL,Undisclosed,25 Mar 2021,2 Apr 2021,1K-15K,NaN,Hal Rogers,...,0.869565,H,Kentucky,5,R,NaN,NaN,NaN,NaN,14.34
103887,American Airlines Group Inc,AAL:US,NaN,BUY,Joint,24 Mar 2021,29 Mar 2021,1K-15K,NaN,Mark Green,...,0.909091,H,Tennessee,7,R,NaN,NaN,NaN,NaN,21.81
103888,United Airlines Holdings Inc,UAL:US,NaN,BUY,Joint,24 Mar 2021,29 Mar 2021,1K-15K,NaN,Mark Green,...,0.909091,H,Tennessee,7,R,NaN,NaN,NaN,NaN,53.83


In [816]:
master_df[master_df["p_party"].isna()]

,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,old_p_full_name,...,fix_name_confidence,p_chamber,p_state,p_district,p_party,cap_gains_over_200_usd,asset_industry,asset_sector,comments,avg_ticker_price


In [817]:
# Ensure names line up
not_fully_confident_name_df = master_df[master_df["fix_name_confidence"] < 1][["old_p_full_name", "p_full_name", "fix_name_confidence"]].drop_duplicates(subset="old_p_full_name").sort_values(by="p_full_name")
to_save_path = "./test.csv"

not_fully_confident_name_df

,old_p_full_name,p_full_name,fix_name_confidence
88155,Aston McEachin,A Donald McEachin,0.774194
40369,Aston Donald McEachin,A Donald McEachin,0.894737
28210,ABIGAIL SPANBERGER,Abigail Davis Spanberger,0.857143
48631,Abigail Spanberger,Abigail Davis Spanberger,0.857143
88149,Adam Schiff,Adam B Schiff,0.916667
...,...,...,...
65986,Greg Steube,W Gregory Steube,0.814815
29198,GREG STEUBE,W Gregory Steube,0.814815
18809,W-GREG STEUBE,W Gregory Steube,0.896552
66020,Bill Keating,William R Keating,0.758621


In [818]:
test_df = master_df.copy(deep=True)

# Fix dates
master_df = master_df[~master_df["tx_date"].isna()]

# Final format is MM/DD/YYYY
def fix_date(in_str:str):
    if not is_str(in_str):
        return pd.NA

    if "Today" in in_str:
        return "03/22/2024"

    in_str = in_str.replace("/3031", "/2021")
    in_str = in_str.replace(",", "")

    # Manual fixes
    if in_str.startswith("0009"):
        return "06/09/2021"

    if in_str.startswith("0021-08"):
        return "08/02/2021"

    if in_str.startswith("0021-06"):
        return "06/22/2021"

    if in_str.startswith("0201"):
        return "06/22/2021"

    if in_str == "0222-11-02":
        return "11/02/2022"

    if in_str == "0222-11-22":
        return "11/22/2022"
    
    if in_str.startswith("20221"):
        in_str = in_str.replace("20221", "2021")

    if in_str.startswith("20222"):
        in_str = in_str.replace("20222", "2022")
    
    # Fix dash
    if "-" in in_str:
        com_list = in_str.split("-")
        in_str =  com_list[1] + "/" + com_list[2] + "/" + com_list[0]
    
    if in_str[-5] != "/" and in_str[-5] != " ":
        in_str = in_str[0:-5] + "/" + in_str[-5:]

    # Fix those with months
    month_list = ["", "JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEPT", "SEP", "OCT", "NOV", "DEC"]
    in_str = in_str.upper()
    for i,month in enumerate(month_list):
        if not month:
            continue
        if month in in_str:
            in_str = in_str.replace(month, "")
            in_str = in_str.replace("  ", " ").strip()
            com = in_str.split(" ")
            day = com[0]
            year = com[1]
    
            m_str = str(i)
            if i > 9:
                m_str = str(i - 1)
            in_str = m_str + "/" + day + "/" + year
            break
        
    
    # Ensure digits line up
    m,d,y = in_str.split("/")
    
    com = in_str.split("/")
    if len(m) == 1:
        m = "0" + m

    if len(d) == 1:
        d = "0" + d

    in_str = m+"/" + d + "/" + y
        
    # Final fixes
    if in_str.endswith("2202"):
        in_str = in_str.removesuffix("2202") + "2020"

    if in_str.endswith("2220"):
        in_str = in_str.removesuffix("2220") + "2020"

    if in_str.endswith("0022"):
        in_str = in_str.removesuffix("0022") + "2022"

    if in_str.endswith("0023"):
        in_str = in_str.removesuffix("0023") + "2023"
    
    
    return in_str

test_df["tx_date"] = test_df["tx_date"].apply(fix_date) 
test_df["notif_date"] = test_df["notif_date"].apply(fix_date) 

test_df["tx_date"] = pd.to_datetime(test_df["tx_date"])
test_df["notif_date"] = pd.to_datetime(test_df["notif_date"])

master_df = test_df
master_df

,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,old_p_full_name,...,fix_name_confidence,p_chamber,p_state,p_district,p_party,cap_gains_over_200_usd,asset_industry,asset_sector,comments,avg_ticker_price
0,DECATUR ALA CITY BRD ED SPL TAX SCH WTS,<NA>,<NA>,P,JT,2014-07-03,2014-07-03,$15001 - $50000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
1,ETOWAH CNTY ALA BRD ED CAP OUTLAY WTS,<NA>,<NA>,P,JT,2014-04-11,2014-04-11,$1001 - $15000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
2,MOBILE COUNTY ALA BRD SCH COMMRSCAP OUTLAY WTS,<NA>,<NA>,P,JT,2014-04-16,2014-04-16,$1001 - $15000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
3,MORGAN STANLEY CAP TR V GTD CAP SECS (MWO),MWO,<NA>,S,JT,2014-05-05,2014-05-05,$1001 - $15000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
4,PHENIX CITY AL SCH WTS GENL OBLIG- AT,<NA>,<NA>,P,JT,2014-03-28,2014-03-28,$15001 - $50000,,MR-MO BROOKS,...,1.000000,H,Alabama,5,R,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103885,BRIDGE BUILDER LARGE CAP VALUE FUND,BBVLX:US,NaN,SELL,Undisclosed,2021-03-25,2021-04-02,1K-15K,NaN,Hal Rogers,...,0.869565,H,Kentucky,5,R,NaN,NaN,NaN,NaN,15.99
103886,BRIDGE BUILDER SMALL MID CAP VALUE FUND,BBVSX:US,NaN,SELL,Undisclosed,2021-03-25,2021-04-02,1K-15K,NaN,Hal Rogers,...,0.869565,H,Kentucky,5,R,NaN,NaN,NaN,NaN,14.34
103887,American Airlines Group Inc,AAL:US,NaN,BUY,Joint,2021-03-24,2021-03-29,1K-15K,NaN,Mark Green,...,0.909091,H,Tennessee,7,R,NaN,NaN,NaN,NaN,21.81
103888,United Airlines Holdings Inc,UAL:US,NaN,BUY,Joint,2021-03-24,2021-03-29,1K-15K,NaN,Mark Green,...,0.909091,H,Tennessee,7,R,NaN,NaN,NaN,NaN,53.83


In [819]:
# Fix owner
test_df = master_df.copy(deep=True)

def fix_owner(in_str: str):
    if not is_str(in_str):
        return pd.NA

    if in_str == "--" or in_str == "Undisclosed":
        return pd.NA
    
    if in_str == "Child" or in_str == "dependent":
        return "DC"

    if in_str == "Joint" or in_str == "joint":
        return "JT"

    if in_str == "Self" or in_str == "self":
        return "SELF"

    if in_str == "Spouse" or in_str == "spouse":
        return "SP"
    
    return in_str

test_df["owner"] = test_df["owner"].apply(fix_owner)


master_df = test_df
master_df.groupby("owner").size()

owner
DC      12536
JT      26879
SELF     4815
SP      24769
dtype: int64

In [820]:
# Fix type
test_df = master_df.copy(deep=True)

PURCHASE_STRING_LIST = [
    "BUY",
    "Purchase",
    "purchase",
    "P"
]
PURCHASE_SYMBOL = "P"

SELL_STRING_LIST = [
    "S",
    "SELL",
    "Sale",
    "sale",
    "sale_full",
    "Sale (Full)"
]
SELL_SYMBOL = "S"

EXCHANGE_STRING_LIST = [
    "E",
    "Exchange",
    "EXCHANGE",
    "exchange"
]
EXCHANGE_SYMBOL = "E"

SELL_PARTIAL_STRING_LIST = [
    "S (PARTIAL)",
    "Sale (Partial)",
    "sale_partial"
]
SELL_PARTIAL_SYMBOL = "S (PARTIAL)"

RECEIVE_STRING_LIST = [
    "RECEIVE"
]
RECEIVE_SYMBOL = "R"


def fix_tx_type(in_str:str):
    if not is_str(in_str):
        return pd.NA
    
    if in_str in PURCHASE_STRING_LIST:
        return PURCHASE_SYMBOL

    if in_str in SELL_STRING_LIST:
        return SELL_SYMBOL

    if in_str in SELL_PARTIAL_STRING_LIST:
        return SELL_PARTIAL_SYMBOL

    if in_str in EXCHANGE_STRING_LIST:
        return EXCHANGE_SYMBOL

    if in_str in RECEIVE_STRING_LIST:
        return RECEIVE_SYMBOL
    
    return in_str

    
test_df["tx_type"] = test_df["tx_type"].apply(fix_tx_type)    
test_df = test_df[~test_df["tx_type"].isna()]

master_df = test_df
master_df.groupby("tx_type", dropna=False).size()

tx_type
E                788
P              52271
R                 61
S              41328
S (PARTIAL)     8637
dtype: int64

In [821]:
# Fix asset_type
test_df = master_df.copy(deep=True)

GS_SYMBOL = "Government Securities and Agency Debt"
GS_STRING_LIST = [
    "GS",
    "Municipal Security",
    GS_SYMBOL
]


AB_SYMBOL = "Asset-Backed Securities"
AB_STRING_LIST = [
    "AB",
    AB_SYMBOL
]

CS_SYMBOL = "Corporate Securities"
CS_STRING_LIST = [
    "CS",
    "Corporate Bond",
    CS_SYMBOL
]

FU_SYMBOL = "Futures"
FU_STRING_LIST = [
    "FU",
    "Commodities/Futures Contract",
    FU_SYMBOL
]

CT_SYMBOL = "Crypto"
CT_STRING_LIST = [
    "CT",
    "Cryptocurrency",
    CT_SYMBOL
]

ET_SYMBOL = "Exchange Traded Notes"
ET_STRING_LIST = [
    "ET",
    ET_SYMBOL
]

HN_SYMBOL = "Hedge Funds"
HN_STRING_LIST = [
    "HN",
    HN_SYMBOL
]

PS_SYMBOL = "Non Public Stock"
PS_STRING_LIST = [
    "PS",
    "Non-Public Stock",
    PS_SYMBOL
]

OI_SYMBOL = "Ownership Interest (Holding Investments)"
OI_STRING_LIST = [
    "O",
    "OI",
    OI_SYMBOL
]

OL_SYMBOL = "Ownership Interest (Engaged in a Trade or Business)"
OL_STRING_LIST = [
    "OL",
    OL_SYMBOL
]

OP_SYMBOL = "Options"
OP_STRING_LIST = [
    "OP",
    "Stock Option",
    OP_SYMBOL
]

OT_SYMBOL = "Other"
OT_STRING_LIST = [
    "OT",
    "Other Securities",
    OT_SYMBOL
]

SA_SYMBOL = "Stock Appreciation"
SA_STRING_LIST = [
    "SA",
    SA_SYMBOL
]

VA_SYMBOL = "Variable Annuity"
VA_STRING_LIST = [
    "VA",
    VA_SYMBOL
]

ST_SYMBOL = "Stock"
ST_STRING_LIST = [
    "ST",
    "Stock",
    ST_SYMBOL
]



def fix_asset_type(in_str:str):
    if not is_str(in_str):
        return OT_SYMBOL
    
    in_str = in_str.replace(".", "")

    if in_str in GS_STRING_LIST:
        return GS_SYMBOL
        
    if in_str in AB_STRING_LIST:
        return AB_SYMBOL

    if in_str in CS_STRING_LIST:
        return CS_SYMBOL

    if in_str in FU_STRING_LIST:
        return FU_SYMBOL

    if in_str in CT_STRING_LIST:
        return CT_SYMBOL
    
    if in_str in ET_STRING_LIST:
        return ET_SYMBOL

    if in_str in HN_STRING_LIST:
        return HN_SYMBOL

    if in_str in PS_STRING_LIST:
        return PS_SYMBOL

    if in_str in OI_STRING_LIST:
        return OI_SYMBOL

    if in_str in OL_STRING_LIST:
        return OL_SYMBOL

    if in_str in OP_STRING_LIST:
        return OP_SYMBOL

    if in_str in OT_STRING_LIST:
        return OT_SYMBOL

    if in_str in SA_STRING_LIST:
        return SA_SYMBOL

    if in_str in VA_STRING_LIST:
        return VA_SYMBOL

    if in_str in ST_STRING_LIST:
        return ST_SYMBOL
    
    return in_str

    
test_df["asset_type"] = test_df["asset_type"].apply(fix_asset_type)    
mask = test_df["data_source"] == "TREND_SPIDER"
test_df.loc[mask, "asset_type"] = "Stock"
master_df = test_df

test_df.groupby("asset_type", dropna=False).size()


asset_type
Asset-Backed Securities                                   36
Corporate Securities                                     905
Crypto                                                     3
Exchange Traded Notes                                      6
Government Securities and Agency Debt                   1018
Hedge Funds                                               63
Non Public Stock                                          74
Options                                                  921
Other                                                  71794
Ownership Interest (Engaged in a Trade or Business)       23
Ownership Interest (Holding Investments)                   3
Stock                                                  28191
Stock Appreciation                                         1
Variable Annuity                                          47
dtype: int64

In [822]:

test_df = master_df.copy(deep=True)
test_df = test_df[~test_df["asset_name"].isna()]
master_df = test_df
master_df[master_df["asset_name"].isna()]

,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,old_p_full_name,...,fix_name_confidence,p_chamber,p_state,p_district,p_party,cap_gains_over_200_usd,asset_industry,asset_sector,comments,avg_ticker_price


In [823]:
# Fix ticker
test_df = master_df.copy(deep=True)

BANNED_TICKER_SUBSTRINGS = [
    "MATURE",
    "DUE ",
    "TREASUR"
]
def fix_ticker(ticker: str):
    if not is_str(ticker) or ticker == "--":
        return pd.NA
    
    upper_str = ticker.upper()
    for substr in BANNED_TICKER_SUBSTRINGS:
        if substr in upper_str:
            return pd.NA
    
    if ticker.count("/") > 1:
        return pd.NA

    if ticker.isnumeric() and len(ticker) < 3:
        return pd.NA
    
    if "%" in ticker:
        return pd.NA

    if "GLAS FUND" in ticker:
        return pd.NA

    ticker = ticker.replace("--\n", "")
    ticker = ticker.replace(",", "")
    ticker = ticker.replace("^", "")
    ticker = ticker.replace("\n", "->")
    if ticker.count(":") == 1:
        ticker = ticker.split(":")[0]

    ticker = ticker.strip()
    ticker = ticker.replace(" ", "")

    if len(ticker) > 7:
        return pd.NA

    return ticker

def get_ticker_location(ticker: str):
    if not is_str(ticker):
        return pd.NA
    
    if ticker.count(":") == 1:
        return ticker.split(":")[1]
        
    return pd.NA
    

test_df["ticker_location"] = test_df["ticker"].apply(get_ticker_location)
test_df["ticker"] = test_df["ticker"].apply(fix_ticker)

mask = test_df["asset_name"].str.upper().str.contains("TREASUR") | test_df["asset_name"].str.upper().str.contains("T BILL") | test_df["asset_name"].str.upper().str.contains("DUE ")

mask2 =  ~test_df["asset_type"].isin(["Stock", "Options", "Crypto", "Other"]) 
mask3 = test_df["tx_type"].isin(["P", "S", "S (PARTIAL)"]) | test_df["tx_type"].isna()
mask = mask | mask2 | ~mask3

test_df.loc[mask, "ticker"] = pd.NA

master_df = test_df
master_df.groupby("ticker", dropna=False).size()

ticker
$ADA         5
$ANKR        1
$BAT         1
$BTC        10
$CELO        1
         ...  
ZTS        134
ZU           1
ZUO          4
ZURVY        6
NaN      12944
Length: 4374, dtype: int64

In [824]:
import re

# Fix asset name
test_df = master_df.copy(deep=True)
self_house_mask = test_df["data_source"].str.startswith("SELF_H") 
self_house_df = test_df[self_house_mask]
test_df = test_df[~self_house_mask]

re_enclosed = re.compile(r"[\(\|].*?[\)\|]")
re_spaces = re.compile(" +")
def fix_self_house_name(asset_name: str):
    asset_name =  re.sub(re_enclosed, "", asset_name)
    asset_name =  re.sub(re_spaces, " ", asset_name)
    return asset_name

self_house_df["asset_name"] = self_house_df["asset_name"].apply(fix_self_house_name) 

test_df = pd.concat([test_df, self_house_df]).reset_index(drop=True)
test_df
master_df = test_df
master_df

,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,old_p_full_name,...,p_chamber,p_state,p_district,p_party,cap_gains_over_200_usd,asset_industry,asset_sector,comments,avg_ticker_price,ticker_location
0,BP plc,BP,Other,P,JT,2021-09-27,2021-10-04,"$1,001 - $15,000",NaN,Virginia Foxx,...,H,North Carolina,5,R,False,Integrated oil Companies,Energy,NaN,NaN,<NA>
1,Exxon Mobil Corporation,XOM,Other,P,JT,2021-09-13,2021-10-04,"$1,001 - $15,000",NaN,Virginia Foxx,...,H,North Carolina,5,R,False,Integrated oil Companies,Energy,NaN,NaN,<NA>
2,Industrial Logistics Properties Trust - Common...,ILPT,Other,P,JT,2021-09-10,2021-10-04,"$15,001 - $50,000",NaN,Virginia Foxx,...,H,North Carolina,5,R,False,Real Estate Investment Trusts,Real Estate,NaN,NaN,<NA>
3,Phillip Morris International Inc,PM,Other,P,JT,2021-09-28,2021-10-04,"$15,001 - $50,000",NaN,Virginia Foxx,...,H,North Carolina,5,R,False,Farming/Seeds/Milling,Consumer Non-Durables,NaN,NaN,<NA>
4,BlackRock Inc,BLK,Other,S (PARTIAL),SELF,2021-09-17,2021-10-04,"$1,001 - $15,000",NaN,Alan S. Lowenthal,...,H,California,47,D,False,Investment Bankers/Brokers/Service,Finance,NaN,NaN,<NA>
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103076,PRINCE GEORGES CNTY MD GO CONSOLIDATED 5.00 DU...,<NA>,Government Securities and Agency Debt,S,JT,2024-02-01,2024-02-01,$500001 - $1000000,,SUZAN-K DELBENE,...,H,Washington,1,D,NaN,NaN,NaN,NaN,NaN,<NA>
103077,BRISTOL-MYERS SQUIBB COMPANY,BMY,Stock,S,<NA>,2024-01-09,2024-01-12,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSEN...,RICK LARSEN,...,H,Washington,2,D,NaN,NaN,NaN,NaN,NaN,<NA>
103078,COLGATE-PALMOLIVE COMPANY,CL,Stock,P,<NA>,2024-01-09,2024-01-12,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSEN...,RICK LARSEN,...,H,Washington,2,D,NaN,NaN,NaN,NaN,NaN,<NA>
103079,THE HERSHEY COMPANY,HSY,Stock,S,<NA>,2024-01-09,2024-01-12,$1001 - $15000,SUBHOLDING OF RICHARD R LARSEN IRA RICK LARSEN...,RICK LARSEN,...,H,Washington,2,D,NaN,NaN,NaN,NaN,NaN,<NA>


In [825]:
# Fix amount
test_df = master_df.copy(deep=True)


STANDARD_AMOUNTS = [
    "1000000-5000000",
    "500000-1000000",
    "250000-500000",
    "100000-250000",
    "50000-100000",
    "15000-50000",
    "1000-15000"
]

def fix_amount(in_amount: str):
    if not is_str(in_amount):
        return pd.NA
    
    in_amount = in_amount.replace("$", "")
    in_amount = in_amount.replace(" ", "")
    in_amount = in_amount.replace(",", "")
    in_amount = in_amount.replace("K", "000")
    in_amount = in_amount.replace("M", "000000")
    in_amount = in_amount.replace("001", "000")

    in_amount = in_amount.upper()
    if "SPOUSE/DC" in in_amount:
        return "1000000+"
    
    if "OVER" in in_amount:
        return "50000000+"

    if in_amount == "250000-500000-100000":
        return "250000-500000"
    
    for amt in STANDARD_AMOUNTS:
        low_amt = amt.split("-")[0]
        high_amt = amt.split("-")[1]
        if in_amount.startswith(low_amt):
            return low_amt + '-' + high_amt
        
        if in_amount.endswith("-" + high_amt):
            return low_amt + '-' + high_amt
    
    if "-" in in_amount:
        in_amount = in_amount.split("-")[0]
    

    return in_amount

def get_amount_max(in_amount:str):
    if "-" in in_amount:
        return float(in_amount.split("-")[1])
    if "<1000" in in_amount:
        return 1000.0
    if "+" in in_amount:
        in_amount = in_amount.replace("+","")
    return float(in_amount)

def get_amount_min(in_amount:str):
    if "-" in in_amount:
        return float(in_amount.split("-")[0]) + 1
    if "<1000" in in_amount:
        return 0.0
    if "+" in in_amount:
        in_amount = in_amount.replace("+","")
    return float(in_amount)
    
def get_amount_avg(in_amount:str):
    if "-" in in_amount:
        max_amt = in_amount.split("-")[1]
        min_amt = in_amount.split("-")[0]
        return (float(max_amt)  + 1 + float(min_amt))/2
    if "<1000" in in_amount:
        return 500.0
    if "+" in in_amount:
        in_amount = in_amount.replace("+","")
    return float(in_amount)

    
    
test_df["amount"] = test_df["amount"].apply(fix_amount)
test_df = test_df[~test_df["amount"].isna()]
test_df["amount_max"] = test_df["amount"].apply(get_amount_max)
test_df["amount_min"] = test_df["amount"].apply(get_amount_min)
test_df["amount_avg"] = test_df["amount"].apply(get_amount_avg)

test_df.groupby("amount_avg",dropna=False).size().sort_values().to_csv("./test.csv")

master_df = test_df
master_df[["doc_id", "data_source", "amount", "tx_type"]]

,doc_id,data_source,amount,tx_type
0,20019557,WATCHER_HOUSE,1000-15000,P
1,20019557,WATCHER_HOUSE,1000-15000,P
2,20019557,WATCHER_HOUSE,15000-50000,P
3,20019557,WATCHER_HOUSE,15000-50000,P
4,20019570,WATCHER_HOUSE,1000-15000,S (PARTIAL)
...,...,...,...,...
103076,20024495,SELF_HOUSE,500000-1000000,S
103077,20024339,SELF_HOUSE,1000-15000,S
103078,20024339,SELF_HOUSE,1000-15000,P
103079,20024339,SELF_HOUSE,1000-15000,S


In [826]:
test_df = master_df.copy(deep=True)
CT_CUTOFF_DATE = "03/24/2021"
mask = (test_df["data_source"] == "CAPITOL_TRADES") | (test_df["tx_date"] < CT_CUTOFF_DATE)
test_df = test_df[mask]

WATCHER_HOUSE_CUTOFF_DATE = "03/13/2013"
mask = (test_df["data_source"].isin(["WATCHER_HOUSE","CAPITOL_TRADES"])) | (test_df["p_chamber"] == "S") | (test_df["tx_date"] < WATCHER_HOUSE_CUTOFF_DATE)
test_df = test_df[mask]

test_df = test_df.sort_values(by="tx_date").reset_index(drop=True)
test_df.to_csv("./test.csv")

master_df = test_df
master_df

,asset_name,ticker,asset_type,tx_type,owner,tx_date,notif_date,amount,asset_desc,old_p_full_name,...,p_party,cap_gains_over_200_usd,asset_industry,asset_sector,comments,avg_ticker_price,ticker_location,amount_max,amount_min,amount_avg
0,E,EP$C,Other,S,SP,2012-02-27,2012-02-27,1000-15000,,MR-ALAN-S LOWENTHAL,...,D,NaN,NaN,NaN,NaN,NaN,<NA>,15000.0,1001.0,8000.5
1,E,EP$C,Other,S,SP,2012-03-20,2012-03-20,1000-15000,,MR-ALAN-S LOWENTHAL,...,D,NaN,NaN,NaN,NaN,NaN,<NA>,15000.0,1001.0,8000.5
2,KANSAS CITY SOUTHERN,KSU,Other,P,SP,2012-06-06,2012-06-06,1000-15000,,MR-ALAN-S LOWENTHAL,...,D,NaN,NaN,NaN,NaN,NaN,<NA>,15000.0,1001.0,8000.5
3,BioLife Solutions Inc,BLFSD,Other,P,<NA>,2012-06-19,2021-08-26,1000-15000,NaN,Tom Malinowski,...,D,False,<NA>,<NA>,NaN,NaN,<NA>,15000.0,1001.0,8000.5
4,PROCTER GAMBLE COMPANY,PG,Other,S,JT,2012-07-24,2012-10-05,1000-15000,SUBHOLDING OF BROKERAGE 2 USAA 8425,MS-TAMMY DUCKWORTH,...,D,NaN,NaN,NaN,NaN,NaN,<NA>,15000.0,1001.0,8000.5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54935,US TREASURY BOND,<NA>,Other,P,SP,2024-03-07,2024-03-08,15000-50000,NaN,Pete Sessions,...,R,NaN,NaN,NaN,NaN,<NA>,<NA>,50000.0,15001.0,32500.5
54936,Capital One Financial Corp,COF,Other,P,<NA>,2024-03-07,2024-03-13,1000-15000,NaN,Bill Keating,...,D,NaN,NaN,NaN,NaN,<NA>,US,15000.0,1001.0,8000.5
54937,US TREASURY BOND,<NA>,Other,P,<NA>,2024-03-08,2024-03-13,250000-500000,NaN,Mike Garcia,...,R,NaN,NaN,NaN,NaN,<NA>,<NA>,500000.0,250001.0,375000.5
54938,NGL Energy Partners LP,NGL,Other,S,<NA>,2024-03-11,2024-03-14,15000-50000,NaN,Mark Green,...,R,NaN,NaN,NaN,NaN,5.9,US,50000.0,15001.0,32500.5


In [827]:
# Select columns for final export
master_df = master_df[[
    "p_full_name",
    "p_chamber",
    "p_state",
    "p_district",
    "p_party",
    "asset_name",
    "ticker",
    "ticker_location",
    "tx_type",
    "owner",
    "tx_date",
    "notif_date",
    "amount",
    "asset_desc",
    "amount_min",
    "amount_avg",
    "amount_max",
    "doc_id",
    "data_source"
]]

master_df.to_csv("master.csv", encoding="utf-8", index=False)
master_df

,p_full_name,p_chamber,p_state,p_district,p_party,asset_name,ticker,ticker_location,tx_type,owner,tx_date,notif_date,amount,asset_desc,amount_min,amount_avg,amount_max,doc_id,data_source
0,Alan S Lowenthal,H,California,47,D,E,EP$C,<NA>,S,SP,2012-02-27,2012-02-27,1000-15000,,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
1,Alan S Lowenthal,H,California,47,D,E,EP$C,<NA>,S,SP,2012-03-20,2012-03-20,1000-15000,,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
2,Alan S Lowenthal,H,California,47,D,KANSAS CITY SOUTHERN,KSU,<NA>,P,SP,2012-06-06,2012-06-06,1000-15000,,1001.0,8000.5,15000.0,20000945,SELF_HOUSE
3,Tom Malinowski,H,New Jersey,7,D,BioLife Solutions Inc,BLFSD,<NA>,P,<NA>,2012-06-19,2021-08-26,1000-15000,NaN,1001.0,8000.5,15000.0,20019374,WATCHER_HOUSE
4,Tammy Duckworth,S,Illinois,<NA>,D,PROCTER GAMBLE COMPANY,PG,<NA>,S,JT,2012-07-24,2012-10-05,1000-15000,SUBHOLDING OF BROKERAGE 2 USAA 8425,1001.0,8000.5,15000.0,20000923,SELF_HOUSE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54935,Pete Sessions,H,Texas,17,R,US TREASURY BOND,<NA>,<NA>,P,SP,2024-03-07,2024-03-08,15000-50000,NaN,15001.0,32500.5,50000.0,NaN,CAPITOL_TRADES
54936,William R Keating,H,Massachusetts,9,D,Capital One Financial Corp,COF,US,P,<NA>,2024-03-07,2024-03-13,1000-15000,NaN,1001.0,8000.5,15000.0,NaN,CAPITOL_TRADES
54937,Mike Garcia,H,California,27,R,US TREASURY BOND,<NA>,<NA>,P,<NA>,2024-03-08,2024-03-13,250000-500000,NaN,250001.0,375000.5,500000.0,NaN,CAPITOL_TRADES
54938,Mark E Green,H,Tennessee,7,R,NGL Energy Partners LP,NGL,US,S,<NA>,2024-03-11,2024-03-14,15000-50000,NaN,15001.0,32500.5,50000.0,NaN,CAPITOL_TRADES
